# autofollowdown on a GPU — NNI · llm-compressor · NVIDIA ModelOpt

This notebook actually **installs and runs the connected backends** on a GPU. It's built for **Google Colab with a T4** (free tier).

> **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

A T4 (Turing) supports INT8, so NVIDIA ModelOpt INT8 + llm-compressor 4-bit GPTQ both run here. (FP8/FP4 need newer GPUs.) These cells skip cleanly if no GPU is present.

Repo: https://github.com/peetwan/autofollowdown

## 1. Install autofollowdown + all backends

In [ ]:
# autofollowdown (from PyPI once published; from GitHub today)
!pip install -q "git+https://github.com/peetwan/autofollowdown#egg=autofollowdown[examples]"
# the optional, specialized backends:
!pip install -q nni "nvidia-modelopt[torch]" llmcompressor

## 2. Confirm the GPU and that the backends are now connected

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

from autofollowdown import all_backends
print()
for b in all_backends():
    print(f'{b.name:40} {"installed ✓" if b.is_available() else "missing"}')

## 3. Microsoft NNI — structured pruning + ModelSpeedup (real channel removal)

NNI's `ModelSpeedup` physically removes pruned filters, so the parameter count actually drops (not just zeros). One line via `compress_with`.

In [ ]:
import copy, torch, torch.nn as nn
from autofollowdown import compress_with

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10))
    def forward(self, x): return self.net(x)

cnn = CNN()
before = sum(p.numel() for p in cnn.parameters())
try:
    compress_with(cnn, 'nni', dummy_input=torch.randn(1, 3, 32, 32), sparsity=0.5)
    after = sum(p.numel() for p in cnn.parameters())
    print(f'NNI structured pruning + ModelSpeedup: params {before:,} -> {after:,} '
          f'({before/max(after,1):.2f}x fewer)')
except Exception as e:
    print('NNI demo skipped:', e)

## 4. llm-compressor (vLLM) — 4-bit GPTQ (W4A16) on Qwen

Real weight-only 4-bit quantization with GPTQ calibration — the format you'd serve in vLLM. Runs on the T4.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from autofollowdown import compress_with, model_disk_size_mb

LLM_ID = 'Qwen/Qwen3-0.6B'   # try Qwen2.5-1.5B / 3B too
try:
    from llmcompressor.modifiers.gptq import GPTQModifier
    model = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype='auto', device_map='cuda')
    tok = AutoTokenizer.from_pretrained(LLM_ID)
    before_mb = model_disk_size_mb(model)
    compress_with(model, 'llmcompressor',
                  recipe=GPTQModifier(targets='Linear', scheme='W4A16', ignore=['lm_head']),
                  dataset='open_platypus', num_calibration_samples=128)
    model.save_pretrained('qwen3-0.6b-w4a16')
    tok.save_pretrained('qwen3-0.6b-w4a16')
    print(f'GPTQ W4A16 done ✓  fp16 ~{before_mb:.0f} MB -> 4-bit checkpoint in qwen3-0.6b-w4a16/')
    print('Serve it with:  vllm serve qwen3-0.6b-w4a16')
except Exception as e:
    print('llm-compressor demo skipped:', e)

## 5. NVIDIA TensorRT Model Optimizer — INT8 SmoothQuant PTQ on Qwen

Calibrated INT8 SmoothQuant via `mtq.quantize` (the T4 supports INT8). The quantized model can then be exported to TensorRT for fast serving.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from autofollowdown import compress_with

try:
    import modelopt.torch.quantization as mtq
    tok = AutoTokenizer.from_pretrained(LLM_ID)
    m = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype=torch.float16, device_map='cuda')
    texts = ['The quick brown fox jumps over the lazy dog.',
             'Large language models can be compressed with quantization.',
             'MMMU is a multimodal benchmark for vision-language models.']
    calib = [tok(t, return_tensors='pt').input_ids.cuda() for t in texts]
    compress_with(m, 'modelopt', calibration_data=calib, quant_cfg=mtq.INT8_SMOOTHQUANT_CFG)
    print('ModelOpt INT8 SmoothQuant PTQ done ✓')
    mtq.print_quant_summary(m)
except Exception as e:
    print('ModelOpt demo skipped:', e)

## 6. Measure the result — WikiText-2 perplexity

Use autofollowdown's perplexity to compare a compressed model against the baseline (same metric the LLM papers report).

In [ ]:
from autofollowdown import evaluate_perplexity, load_wikitext2
try:
    text = load_wikitext2('test')[:3000]
    ppl = evaluate_perplexity(m, tok, text, stride=512, max_length=1024, device='cuda')
    print(f'WikiText-2 perplexity (INT8 SmoothQuant Qwen): {ppl:.2f}')
except Exception as e:
    print('perplexity skipped:', e)

## What you just did

- Installed and ran **three production compression libraries** through one consistent `compress_with(model, backend)` call.
- **NNI**: structurally pruned a CNN (real channel removal via ModelSpeedup).
- **llm-compressor**: produced a **4-bit GPTQ** Qwen checkpoint, ready for vLLM.
- **NVIDIA ModelOpt**: **INT8 SmoothQuant** PTQ on Qwen, exportable to TensorRT.

Swap `LLM_ID` for a bigger Qwen (`Qwen2.5-1.5B`, `Qwen2.5-3B`, `Qwen3-1.7B`) — a T4 handles models up to a few billion params in 4-bit. For the full toolkit (one-command compress + benchmark + auto-pick, MMMU/MMLU-ProX commands), see [`autofollowdown_demo.ipynb`](autofollowdown_demo.ipynb).